In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import statsmodels.api as sm
from scipy.stats import nbinom
from tqdm import tqdm

import torch
import torch_geometric as pyg

from bipartite_gnn.train_test import GNNTrainer
from baseline_evals.feature_selection import variance_filtering

from bipartite_gnn.model import BipartiteGNN
from bipartite_gnn.train_test import GNNTrainer

In [3]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [4]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [5]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

In [6]:
mrna_X.shape

torch.Size([483, 18975])

In [180]:
import warnings
from statsmodels.tools.sm_exceptions import HessianInversionWarning

def differential_exp_nbnm(expression_vector, var_multiplier=1):
    """
    Estimate the differential expression of a gene using the negative binomial distribution.

    Args:
        expression_vector (np.ndarray): The expression vector of the gene.
        var_multiplier (int): The multiplier for the variance threshold. Default is 1.
    Returns:
        select_mask (np.ndarray): The mask of the selected samples.
    """
    if not isinstance(expression_vector, np.ndarray):
        expression_vector = np.array(expression_vector)

    # ignore all warnings
    with warnings.catch_warnings():
        # for some distributions, the fitting will fail, so ignore warnings for those
        warnings.filterwarnings("ignore")
        res = sm.NegativeBinomial(expression_vector, np.ones_like(expression_vector)).fit(start_params=[1, 1], disp=0)

    mu = np.exp(res.params[0])
    p = 1 / (1 + mu * res.params[1])
    r = mu * p / (1 - p)

    var = r * (1-p) / p**2
    # var = np.sqrt(var)
    # std = var # np.sqrt(var)
    mask_above = expression_vector > mu + var * var_multiplier
    mask_below = expression_vector < mu - var * var_multiplier

    return mask_below, mask_above

In [109]:
# select 400 top genes for mrna
select_mask = variance_filtering(mrna_X.numpy(), 1000)
mrna_X_400 = mrna_X[:, select_mask]

# select 400 top genes for cna
select_mask = variance_filtering(cna_X.numpy(), 400)
cna_X_400 = cna_X[:, select_mask].float()

# select 200 top genes for mirna
select_mask = variance_filtering(mirna_X.numpy(), 200)
mirna_X_200 = mirna_X[:, select_mask]

In [28]:
# get interaction data
torch.triu(torch.ones(5,5), diagonal=0)

tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])

In [147]:
# create hetero-data object
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo
import torch_geometric.transforms as T

data = pyg.data.HeteroData()

proj_dim = 128

# sample node features
data["mrna"].x = mrna_X_400.float()
# data["cna"].x = cna_X_400.float()
# data["mirna"].x = mirna_X_200.float()

data["mrna"].y = torch.tensor(y)
# data["mrna"].y = data["cna"].y = data["mirna"].y = torch.tensor(y)

data.omics = ["mrna"] #, "cna", "mirna"]

# feature node projection
data["mrna_feature"].x = torch.ones(mrna_X_400.shape[1], proj_dim)
# data["cna_feature"].x = torch.ones(cna_X_400.shape[1], proj_dim)
# data["mirna_feature"].x = torch.ones(mirna_X_200.shape[1], proj_dim)

# create edges
data.num_relations = 2 # 3 for forward egdes, 3 for backward edges

data["mrna", "diff_exp", "mrna_feature"].edge_index = dense_to_coo(create_diff_exp_connections_nbnom(mrna_X_400))
# data["cna", "diff_exp", "cna_feature"].edge_index = dense_to_coo(create_diff_exp_connections_norm(cna_X_400, 2.0))
# data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(create_diff_exp_connections_norm(mirna_X_200, 2.0))

data = T.ToUndirected()(data)
data = T.AddSelfLoops()(data)

# create train test masks
from sklearn.model_selection import train_test_split

train_mask, test_mask = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=3)
val_mask, test_mask = train_test_split(test_mask, test_size=0.5, random_state=3)

data["train_mask"] = torch.tensor(train_mask)
data["val_mask"] = torch.tensor(val_mask)
data["test_mask"] = torch.tensor(test_mask)

data

HeteroData(
  omics=[1],
  num_relations=2,
  train_mask=[386],
  val_mask=[48],
  test_mask=[49],
  mrna={
    x=[483, 1000],
    y=[483],
  },
  mrna_feature={ x=[1000, 128] },
  (mrna, diff_exp, mrna_feature)={ edge_index=[2, 8702] },
  (mrna_feature, rev_diff_exp, mrna)={ edge_index=[2, 8702] }
)

In [148]:
data.train_mask, data.val_mask, data.test_mask

(tensor([ 44, 393, 317, 251,  33, 374, 231, 324, 370, 240, 277, 138, 246, 217,
           3, 442,  83, 339, 460, 431,  12,  48, 434, 443, 278, 425, 252,  55,
         347, 298, 146, 478, 166, 314, 174,  71, 228, 107, 439, 356,  76, 132,
         152, 368, 149, 161, 133, 416, 445, 390,  14, 381, 151, 241, 100, 305,
         334, 211, 113,  17, 250, 194, 120, 413,  88, 125, 399, 303, 401,  85,
         354, 201, 357, 476, 271, 172, 466, 408, 202,  77, 360, 410, 325, 395,
          42, 232, 355,  80, 117, 183, 327, 279, 455, 296, 197,  49, 322,  40,
         373, 139, 126, 418, 391, 402, 103, 248, 178,  32, 169, 378, 306,  18,
         336, 111, 281, 229, 160, 128, 464, 400, 207,  39, 441, 438, 233, 230,
         242, 285, 320, 428, 394, 436, 245, 273, 313, 465, 129, 411, 130, 181,
         419, 195, 430, 266, 312, 367, 467, 276, 407, 170, 127,  16, 255, 180,
         189, 427, 209, 140, 341, 315, 291, 134, 309,  57, 308, 380,  41, 258,
         282, 468,  34,  91,  11,  62,  78, 365, 177

In [149]:
model = BipartiteGNN(
    input_sizes=[mrna_X_400.shape[1]], # , cna_X_400.shape[1], mirna_X_200.shape[1]],
    proj_dim=proj_dim,
    # proj_dropout=0.5,
    hidden_channels=[proj_dim, 64, 64], # num_layers = len(hidden_channels) - 1
    num_labels=len(np.unique(y)),
    num_relations=data.num_relations,
    num_bases=data.num_relations,
    num_heads=1,
    dropout=0.0,
    feature_integration_mode="linear",
)

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4),
    params={
        "l1_lambda" : 0.1,
    }
)

trainer.train(
    data=data,
    n_epochs=5,
    log_interval=1,
)

torch.Size([1, 483, 64])
torch.Size([1, 483, 64])
torch.Size([1, 483, 64])
Epoch: 001, 
Train Loss: 203.9625, Train Acc: 0.2513, Train F1 Macro: 0.1004, Train F1 Weighted: 0.1009
Val Loss: 2.2819, Val Acc: 0.5208, Val F1 Macro: 0.1712, Val F1 Weighted: 0.3567
Test Loss: 4.6394, Test Acc: 0.3878, Test F1 Macro: 0.1397, Test F1 Weighted: 0.2167
##################################################
torch.Size([1, 483, 64])
torch.Size([1, 483, 64])
torch.Size([1, 483, 64])
Epoch: 002, 
Train Loss: 195.3940, Train Acc: 0.4560, Train F1 Macro: 0.1566, Train F1 Weighted: 0.2856
Val Loss: 2.0541, Val Acc: 0.2500, Val F1 Macro: 0.1000, Val F1 Weighted: 0.1000
Test Loss: 3.9940, Test Acc: 0.1224, Test F1 Macro: 0.0545, Test F1 Weighted: 0.0267
##################################################
torch.Size([1, 483, 64])
torch.Size([1, 483, 64])
torch.Size([1, 483, 64])
Epoch: 003, 
Train Loss: 184.2142, Train Acc: 0.1788, Train F1 Macro: 0.0758, Train F1 Weighted: 0.0542
Val Loss: 2.1984, Val Acc: 0.

In [ ]:
model(data.clone()).shape

Data(omics=[3], num_relations=6, edge_index=[2, 13842], x=[2449, 128], y=[2449], node_type=[2449], edge_type=[13842])
torch.Size([3, 483, 64])


torch.Size([483])

In [11]:
dh = data.to_homogeneous()

HeteroData(
  num_relations=6,
  mrna={
    x=[483, 400],
    y=[483],
  },
  cna={
    x=[483, 400],
    y=[483],
  },
  mirna={
    x=[483, 200],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] },
  (mrna, diff_exp, mrna_feature)={ edge_index=[2, 3730] },
  (cna, diff_exp, cna_feature)={ edge_index=[2, 1997] },
  (mirna, diff_exp, mirna_feature)={ edge_index=[2, 1194] },
  (mrna_feature, rev_diff_exp, mrna)={ edge_index=[2, 3730] },
  (cna_feature, rev_diff_exp, cna)={ edge_index=[2, 1997] },
  (mirna_feature, rev_diff_exp, mirna)={ edge_index=[2, 1194] }
)

In [12]:
dh

Data(num_relations=6, edge_index=[2, 13842], x=[45431, 400], y=[45431], node_type=[45431], edge_type=[13842])

In [47]:
data

HeteroData(
  mrna={
    x=[483, 18975],
    y=[483],
  },
  cna={
    x=[483, 24776],
    y=[483],
  },
  mirna={
    x=[483, 231],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] }
)

In [48]:
mrna_X.shape, cna_X.shape, mirna_X.shape

(torch.Size([483, 18975]), torch.Size([483, 24776]), torch.Size([483, 231]))

In [49]:
dh

Data(x=[45431, 24776], y=[45431], node_type=[45431])

In [50]:
# get counts of unique values in the dh.node_type tensor
node_type_counts = torch.unique(dh.node_type, return_counts=True)
node_type_counts

(tensor([0, 1, 2, 3, 4, 5]),
 tensor([  483,   483,   483, 18975, 24776,   231]))

In [19]:
data

HeteroData(
  mrna={
    x=[18975, 483],
    y=[483],
  },
  cna={
    x=[24776, 483],
    y=[483],
  },
  mirna={
    x=[231, 483],
    y=[483],
  }
)

In [ ]:
# train the model